# Carga de imagenes al datastore y mosaic dataset

Flujo operativo posterior a la preparacion del notebook `008`. Usa `04_ready_for_datastore.csv` para copiar archivos al datastore y `06_attribute_updates.csv` para actualizar atributos del mosaic dataset.

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib

import pandas as pd

import core.mosaic_loader as mosaic_loader
mosaic_loader = importlib.reload(mosaic_loader)
from core.mosaic_loader import *

# PARAMETROS
RUN_PREPARACION = "20260615_155625"
OUTPUT_PREPARACION_DIR = Path.cwd() / "outputs" / "preparacion_carga_mosaico" / RUN_PREPARACION

READY_FOR_DATASTORE_CSV = OUTPUT_PREPARACION_DIR / "04_ready_for_datastore.csv"
ATTRIBUTE_UPDATES_CSV = OUTPUT_PREPARACION_DIR / "06_attribute_updates.csv"

PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

# Seguridad operacional: validar primero con DRY_RUN=True. Cambiar a False para ejecutar copia/carga/update.
DRY_RUN = False
OVERWRITE_COPY = False
SKIP_EXISTING_MOSAIC_NAME = True

# Valores fijos definidos para esta carga.
MAXPS_VALUE = 10000
LOWPS_VALUE = 0.15

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path.cwd() / "outputs" / "carga_mosaico" / run_timestamp
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Preparacion:", OUTPUT_PREPARACION_DIR)
print("CSV carga:", READY_FOR_DATASTORE_CSV)
print("CSV atributos:", ATTRIBUTE_UPDATES_CSV)
print("Mosaic dataset:", PATH_MOSAIC_DATASET)
print("Salida:", OUTPUT_DIR)
print("DRY_RUN:", DRY_RUN)

## 1. Cargar manifiestos

Se valida que cada imagen lista tenga atributos asociados antes de ejecutar cualquier operacion.

In [ ]:
load_df = load_ready_and_attributes(READY_FOR_DATASTORE_CSV, ATTRIBUTE_UPDATES_CSV)

required_columns = ["path", "destination_path", "Name", "Sector", "Fecha_Adqui", "URL", "Proyecto", "Sensor", "Fecha_Publ"]
missing_columns = [column for column in required_columns if column not in load_df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

validation_summary = pd.DataFrame(
    [
        {"metric": "records_to_process", "value": len(load_df)},
        {"metric": "missing_source_path", "value": int((~load_df["path"].map(lambda value: Path(value).exists())).sum())},
        {"metric": "missing_destination_path", "value": int(load_df["destination_path"].isna().sum())},
        {"metric": "unique_destination_paths", "value": int(load_df["destination_path"].nunique())},
        {"metric": "unique_names", "value": int(load_df["Name"].nunique())},
    ]
)

display(validation_summary)
display(load_df[["file_name", "Name", "destination_path", "Sector", "Fecha_Adqui", "Proyecto", "Sensor", "Fecha_Publ"]].head(20))

## 2. Ejecutar copia, carga al mosaico, footprints y atributos

Con `DRY_RUN=True` no escribe archivos ni modifica el mosaic dataset. Con `DRY_RUN=False` ejecuta el flujo completo por imagen.

In [ ]:
results = []

for index, row in load_df.iterrows():
    print(f"[{index + 1}/{len(load_df)}] {row['Name']}")
    result = process_mosaic_load_row(
        row,
        PATH_MOSAIC_DATASET,
        overwrite_copy=OVERWRITE_COPY,
        skip_existing_mosaic_name=SKIP_EXISTING_MOSAIC_NAME,
        maxps_value=MAXPS_VALUE,
        lowps_value=LOWPS_VALUE,
        dry_run=DRY_RUN,
    )
    results.append(result)

results_df = pd.DataFrame(results)
display(results_df)
display(results_df["overall_status"].value_counts(dropna=False).reset_index(name="count").rename(columns={"index": "overall_status"}))

## 3. Exportar resultados de ejecucion

In [ ]:
summary_rows = [
    {"metric": "run_timestamp", "value": run_timestamp},
    {"metric": "preparation_run", "value": RUN_PREPARACION},
    {"metric": "dry_run", "value": DRY_RUN},
    {"metric": "mosaic_dataset", "value": PATH_MOSAIC_DATASET},
    {"metric": "records_to_process", "value": len(load_df)},
    {"metric": "maxps_value", "value": MAXPS_VALUE},
    {"metric": "lowps_value", "value": LOWPS_VALUE},
]

for column in ["copy_status", "mosaic_add_status", "footprint_status", "attribute_status", "overall_status"]:
    if column in results_df.columns:
        for status, count in results_df[column].value_counts(dropna=False).items():
            summary_rows.append({"metric": f"{column}_{status}", "value": int(count)})

summary_df = pd.DataFrame(summary_rows)

summary_csv = OUTPUT_DIR / "00_summary.csv"
results_csv = OUTPUT_DIR / "01_load_results.csv"
errors_csv = OUTPUT_DIR / "02_errors.csv"

summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
results_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
error_columns = [column for column in results_df.columns if column.endswith("_error")]
error_filter = results_df[error_columns].notna().any(axis=1) if error_columns else pd.Series(False, index=results_df.index)
results_df[error_filter].to_csv(errors_csv, index=False, encoding="utf-8-sig")

display(summary_df)
print("Resultados exportados en:", OUTPUT_DIR)

In [ ]:
Name = 'CL_MLP_PAO_IF_Ortho_26_05_01_Estacion_Cabecera'
load_df['Name'].tolist()

In [ ]:
q = tuple(results_df['Name'])
q = f'Name IN {q}'
q

In [ ]:
import arcpy

Estado = 'Activo'
with arcpy.da.SearchCursor(
    PATH_MOSAIC_DATASET,
    ['NombreVuelo'],
    where_clause=None,
    sql_clause=(None, "ORDER BY Name ASC")
) as cursor:
    for c in cursor:
        if c[0] is not None:
            print(c)
            break

In [ ]:
# with arcpy.da.UpdateCursor(
#     PATH_MOSAIC_DATASET,
#     ['Estado'],
#     where_clause=q
    
# ) as cursor:
#     for c in cursor:
#         c[0] = 'Activo'
#         cursor.updateRow(c)

## Se cargan los footprint

In [ ]:
arcpy.env.overwriteOutput = True
fc_footprint_temp = 'in_memory/foot3'
arcpy.ExportMosaicDatasetGeometry_management(PATH_MOSAIC_DATASET,
                                             out_feature_class=fc_footprint_temp,
                                             where_clause=q)


In [ ]:
arcpy.GetCount_management(fc_footprint_temp)[0]

In [ ]:
import re
fc_footprint = r'\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'

mapsfields = {
    'FechaAdqui':'Fecha_Adqu', 
    'FechaCarga':'Fecha_Publ' ,
    'NombreVuelo':'Nombre_de_Vuelo', 
    'ProductName':'ProductNam'
    }
## Se renombran los campos de la exportacion
for f in mapsfields:
    old_name = f
    new_name = mapsfields[f]
    print(f'{old_name} ==> {new_name}')
    arcpy.AlterField_management(fc_footprint_temp,old_name,new_name)

In [ ]:
arcpy.Append_management(fc_footprint_temp,fc_footprint,'NO_TEST','')

In [ ]:
len([c for c in arcpy.da.SearchCursor(fc_footprint,['Nombre_de_Vuelo'],"Nombre_de_Vuelo is null")])

In [ ]:
cols = ['Nombre_de_Vuelo','Sector','Fecha_Adqu']
with arcpy.da.UpdateCursor(
    fc_footprint,
    cols,
    where_clause=None,
    sql_clause=(None, "ORDER BY Name ASC")
) as cursor:
    cnt = 0
    for c in cursor:
        if c[1] and c[2]:
            cnt +=1
            fecha = pd.to_datetime(c[2]).strftime('%y_%m_%d')
            numero = str(cnt).zfill(3)
            sector = c[1]
            name_ = f'{numero}_{fecha}_{sector}'
            c[0] = name_
            cursor.updateRow(c)
            print(name_)
       

In [ ]:
## Se actualiza en los footprint
querymosaic = [c[0] for c in arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name'])]

In [ ]:
# querymosaic = f"Name IN {tuple(querymosaic)}"
cursor = arcpy.da.SearchCursor(fc_footprint,['Name','Nombre_de_Vuelo'])
df = pd.DataFrame(cursor,columns=['Name','Nombre_de_Vuelo'])
print(f'total de registros {df.shape[0]}')
df = df[df.Name.isin(querymosaic)]
print(f'total de registros {df.shape[0]}') 
df.head()

In [ ]:
result = []
with arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','NombreVuelo']) as cursor:
    for c in cursor:
        try:
            name = c[0]
            nombre_vuelo = df.loc[df.Name == name,'Nombre_de_Vuelo'].values[0]
            result.append([c[1],nombre_vuelo])
        except:
            continue
        
df2 = pd.DataFrame(result,columns=['NombreVueloOld','NombreVuelo'])
df2.head()

In [ ]:
query_update = tuple(df2.NombreVueloOld)
query_update = f'NombreVuelo IN {query_update}'
dict_rename = dict(zip(df2.NombreVueloOld,df2.NombreVuelo))

In [ ]:
with arcpy.da.UpdateCursor(PATH_MOSAIC_DATASET,['NombreVuelo']) as cursor:
    for c in cursor:
        new_name_ = dict_rename.get(c[0],None)
        if new_name_:
            c[0] = new_name
            cursor.updateRow(c)
            

In [ ]:
aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
 
aprx = arcpy.mp.ArcGISProject(aprx_path)
maps = aprx.listMaps(map_name)
 
if not maps:
    print("No se encontró el mapa")
else:
    m = maps[0]
 
    # 1️⃣ Renombrar
    for lyr in m.listLayers():
        try:
            original_name = lyr.name
            new_name = original_name
            if new_name.startswith(prefix):
                new_name = new_name.replace(prefix, "", 1)
            if new_name.endswith(suffix):
                new_name = new_name[:-len(suffix)]
            if new_name != original_name:
                print(f"Renombrando: {original_name} -> {new_name}")
                lyr.name = new_name
        except Exception as e:
            print(f"Error con capa {lyr.name}: {e}")
 
#     # 2️⃣ Ordenar
#     layers = [lyr for lyr in m.listLayers() if lyr.isFeatureLayer or lyr.isRasterLayer]
#     layers_sorted = sorted(layers, key=lambda l: l.name)
#     for lyr in layers_sorted:
#         m.moveLayer(m.listLayers()[0], lyr, "BEFORE")
#     print("Capas ordenadas")
 
#     # 3️⃣ Guardar una sola vez
#     aprx.save()
#     print("Proyecto guardado")
 
# del aprx  # buena práctica siempre

In [ ]:
maps

In [ ]:
load_df.columns

In [ ]:
load_df.loc[0,'destination_path']

In [ ]:
from pathlib import Path
import csv
import arcpy

LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"

aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
TARGET_GROUP_LAYER_NAME = None  # Opcional: escribir aqui el nombre exacto del grupo si quieres forzarlo.
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
SAVE_COPY_FOR_REVIEW = True
ADD_ONLY_MISSING_TO_GROUP = True  # La lista del CSV se agrega como nuevas capas al grupo existente.
APRX_COPY_PATH = f"{Path.cwd()}\APRX\{Path(aprx_path).stem}_verificacion_carga_imagenes.aprx"

def clean_layer_name(name):
    clean_name = Path(str(name)).name
    if clean_name.startswith(prefix):
        clean_name = clean_name.replace(prefix, "", 1)
    if clean_name.endswith(suffix):
        clean_name = clean_name[:-len(suffix)]
    return clean_name


def read_loaded_image_paths(csv_path):
    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [
            row["destination_path"]
            for row in rows
            if row.get("overall_status") == "ok" and row.get("destination_path")
        ]


def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def is_child_of_group(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    return long_name.startswith(group_long_name + "\\")


def layer_matches_image_naming(layer):
    name = Path(str(layer.name)).name
    return name.startswith(prefix) or name.endswith(suffix) or clean_layer_name(name) != name


def find_target_group_layer(map_obj, explicit_group_name=None):
    groups = [lyr for lyr in map_obj.listLayers() if lyr.isGroupLayer]

    if explicit_group_name:
        matches = [lyr for lyr in groups if lyr.name == explicit_group_name or layer_long_name(lyr) == explicit_group_name]
        if not matches:
            available = [layer_long_name(lyr) for lyr in groups]
            raise ValueError(f"No se encontro el grupo '{explicit_group_name}'. Grupos disponibles: {available}")
        return matches[0]

    candidates = []
    for group_layer in groups:
        child_layers = [lyr for lyr in map_obj.listLayers() if lyr != group_layer and is_child_of_group(lyr, group_layer)]
        image_children = [lyr for lyr in child_layers if (lyr.isRasterLayer or lyr.isFeatureLayer) and layer_matches_image_naming(lyr)]
        if image_children:
            candidates.append((len(image_children), group_layer))

    if not candidates:
        available = [layer_long_name(lyr) for lyr in groups]
        raise ValueError(
            "No se pudo detectar automaticamente el grupo destino. "
            f"Define TARGET_GROUP_LAYER_NAME. Grupos disponibles: {available}"
        )

    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1]


def group_child_layers(map_obj, group_layer):
    return [
        lyr
        for lyr in map_obj.listLayers()
        if lyr != group_layer and is_child_of_group(lyr, group_layer) and (lyr.isFeatureLayer or lyr.isRasterLayer)
    ]


def add_raster_to_group(map_obj, group_layer, image_path, target_name):
    top_layer = map_obj.addDataFromPath(image_path)
    top_layer.name = target_name

    grouped_layers = map_obj.addLayerToGroup(group_layer, top_layer, "BOTTOM")
    map_obj.removeLayer(top_layer)

    if grouped_layers:
        grouped_layer = grouped_layers[0] if isinstance(grouped_layers, list) else grouped_layers
        grouped_layer.name = target_name
        return grouped_layer

    for lyr in group_child_layers(map_obj, group_layer):
        if clean_layer_name(lyr.name).lower() == target_name.lower():
            lyr.name = target_name
            return lyr

    return None


image_paths_to_add = read_loaded_image_paths(LOAD_RESULTS_CSV)
print(f"Imagenes cargadas a revisar en APRX: {len(image_paths_to_add)}")

aprx = arcpy.mp.ArcGISProject(aprx_path)

try:
    maps = aprx.listMaps(map_name)

    if not maps:
        print("No se encontro el mapa")
    else:
        m = maps[0]
        target_group = find_target_group_layer(m, TARGET_GROUP_LAYER_NAME)
        print(f"Grupo destino: {layer_long_name(target_group)}")

        existing_group_layers_before = group_child_layers(m, target_group)
        existing_names = {clean_layer_name(lyr.name).lower() for lyr in existing_group_layers_before}
        print(f"Capas existentes en el grupo antes de agregar: {len(existing_group_layers_before)}")
        added_count = 0
        skipped_count = 0
        error_count = 0

        # 1. Agregar al grupo destino solo las imagenes nuevas del CSV.
        #    Las capas que ya existen en el grupo se conservan y no se reemplazan.
        for image_path in image_paths_to_add:
            target_name = clean_layer_name(Path(image_path).name)
            target_key = target_name.lower()

            if ADD_ONLY_MISSING_TO_GROUP and target_key in existing_names:
                print(f"Ya existe en el grupo, se conserva y se omite: {target_name}")
                skipped_count += 1
                continue

            try:
                grouped_layer = add_raster_to_group(m, target_group, image_path, target_name)
                existing_names.add(target_key)
                added_count += 1
                print(f"Agregada al grupo: {target_name}")
            except Exception as exc:
                error_count += 1
                print(f"Error agregando {image_path}: {exc}")

        # 2. Renombrar solamente las capas del grupo destino con la nomenclatura limpia.
        renamed_count = 0
        for lyr in group_child_layers(m, target_group):
            try:
                original_name = lyr.name
                new_name = clean_layer_name(original_name)
                if new_name != original_name:
                    print(f"Renombrando: {original_name} -> {new_name}")
                    lyr.name = new_name
                    renamed_count += 1
            except Exception as exc:
                print(f"Error con capa {lyr.name}: {exc}")

        # 3. Ordenar alfabeticamente solamente las capas del grupo destino.
        layers = group_child_layers(m, target_group)
        for lyr in sorted(layers, key=lambda layer: layer.name, reverse=True):
            current_group_layers = group_child_layers(m, target_group)
            first_layer = current_group_layers[0] if current_group_layers else None
            if first_layer and lyr != first_layer:
                m.moveLayer(first_layer, lyr, "BEFORE")

        print("Capas del grupo ordenadas")
        existing_group_layers_after = group_child_layers(m, target_group)
        print(f"Capas existentes en el grupo despues de agregar: {len(existing_group_layers_after)}")
        print(f"Resumen APRX: nuevas_agregadas={added_count}, existentes_conservadas={skipped_count}, renombradas={renamed_count}, errores={error_count}")

        # 4. Guardar el proyecto o una copia de revision.
        if SAVE_COPY_FOR_REVIEW:
            aprx.saveACopy(APRX_COPY_PATH)
            print(f"Copia de verificacion guardada: {APRX_COPY_PATH}")
        else:
            aprx.save()
            print("Proyecto original guardado")
finally:
    del aprx




In [ ]:
aprx.saveACopy(APRX_COPY_PATH)

In [ ]:
fc_footprint
q

In [ ]:
PATH_MOSAIC_DATASET
cursor = arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','URL','OBJECTID'],q)
df = pd.DataFrame(cursor,columns=['Name','URL','OBJECTID'])

In [ ]:
with arcpy.da.UpdateCursor(fc_footprint,['Name','URL','ProductNam'],q) as cursor:
    for c in cursor:
    
        # url_ = df.loc[df.Name == c[0],'URL'].values[0]
        # url_
        # c[1] = url_
        obj = df.loc[df.Name == c[0],'OBJECTID'].values[0]
        c[2] = str(obj)
        
        cursor.updateRow(c)

In [ ]:
c

In [ ]:
cursor = arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,['Name','URL','ProductName'],q)
df = pd.DataFrame(cursor,columns=['Name','URL','ProductName'])

In [ ]:
df = df[df.URL.notnull()].reset_index(drop=True)
df.head()

In [ ]:
df.loc[2,'URL']

In [ ]:
'https://sig.aminerals.cl/imgdyn/rest/services/CL_MLP_PAO/CL_MLP_PAO_IF_Ortho_Geosupport/ImageServer/file?id=.\\El_Mauro_Drone\\26_05\\CL_MLP_PAO_IF_Ortho_26_05_14_Camino_Alternativo_Salamanca.tif&rasterId='

In [ ]:
import os
FOLDER_APRX = rf'{Path.cwd()}\APRX'
os.makedirs(FOLDER_APRX,exist_ok=True)

In [ ]:
from pathlib import Path
import csv
import re
import arcpy


LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"

aprx_path = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
map_name = "CL MLP PAO 27 Imagenes Aereas PAO Image Server"
PARENT_GROUP_NAME = "Vuelos Drone PAO"
TARGET_GROUP_NAME = "Imagenes Drone"
prefix = "CL_MLP_PAO_IF_Ortho_"
suffix = ".tif"
SAVE_COPY_FOR_REVIEW = True
ADD_ONLY_MISSING_TO_GROUP = True
APRX_COPY_PATH = fr"{FOLDER_APRX}\{Path(aprx_path).stem}_verificacion_carga_imagenes.aprx"


def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def short_layer_name(name):
    short_name = Path(str(name)).name

    while short_name.startswith("tmp_"):
        short_name = short_name.replace("tmp_", "", 1)

    if short_name.startswith(prefix):
        short_name = short_name.replace(prefix, "", 1)

    if short_name.endswith(suffix):
        short_name = short_name[:-len(suffix)]

    return short_name


def layer_name_from_path(path_value):
    return short_layer_name(Path(str(path_value)).name)


def comparable_layer_keys(name):
    short_name = short_layer_name(name)
    full_name = f"{prefix}{short_name}{suffix}"
    tmp_full_name = f"tmp_{full_name}"
    return {short_name.lower(), full_name.lower(), tmp_full_name.lower()}


def is_image_layer_name(name):
    short_name = short_layer_name(name)
    return bool(re.match(r"^\d{2}_\d{2}_\d{2}_.+", short_name))


def sort_key_newest_first(layer_name):
    name = short_layer_name(layer_name)
    match = re.search(r"^(\d{2})_(\d{2})_(\d{2})_", name)
    if match:
        yy, mm, dd = match.groups()
        return (int(yy), int(mm), int(dd), name.lower())
    return (0, 0, 0, name.lower())


def read_loaded_image_paths(csv_path):
    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [
            row["destination_path"]
            for row in rows
            if row.get("overall_status") == "ok" and row.get("destination_path")
        ]


def is_descendant_of_group(layer, group_layer):
    return layer_long_name(layer).startswith(layer_long_name(group_layer) + "\\")


def is_direct_child_of_group(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    prefix_name = group_long_name + "\\"
    if not long_name.startswith(prefix_name):
        return False
    relative_name = long_name[len(prefix_name):]
    return "\\" not in relative_name


def direct_child_groups(map_obj, parent_group):
    return [
        lyr
        for lyr in map_obj.listLayers()
        if lyr.isGroupLayer and lyr != parent_group and is_direct_child_of_group(lyr, parent_group)
    ]


def group_direct_raster_layers(map_obj, group_layer):
    return [
        lyr
        for lyr in map_obj.listLayers()
        if lyr != group_layer and is_direct_child_of_group(lyr, group_layer) and lyr.isRasterLayer and not lyr.isGroupLayer
    ]


def find_group(map_obj, group_name):
    matches = [lyr for lyr in map_obj.listLayers() if lyr.isGroupLayer and lyr.name == group_name]
    if not matches:
        available = [layer_long_name(lyr) for lyr in map_obj.listLayers() if lyr.isGroupLayer]
        raise ValueError(f"No se encontro el grupo '{group_name}'. Grupos disponibles: {available}")
    if len(matches) > 1:
        print(f"Advertencia: se encontraron {len(matches)} grupos '{group_name}'. Se usara: {layer_long_name(matches[0])}")
    return matches[0]


def find_or_restore_target_group(map_obj, parent_group):
    expected_long_name = f"{layer_long_name(parent_group)}\\{TARGET_GROUP_NAME}"
    exact_matches = [
        lyr
        for lyr in direct_child_groups(map_obj, parent_group)
        if lyr.name == TARGET_GROUP_NAME or layer_long_name(lyr) == expected_long_name
    ]
    if exact_matches:
        return exact_matches[0]

    candidates = []
    for group in direct_child_groups(map_obj, parent_group):
        raster_children = group_direct_raster_layers(map_obj, group)
        image_children = [lyr for lyr in raster_children if is_image_layer_name(lyr.name)]
        if image_children:
            candidates.append((len(image_children), group))

    if candidates:
        candidates.sort(key=lambda item: item[0], reverse=True)
        target_group = candidates[0][1]
        original_group_name = target_group.name
        target_group.name = TARGET_GROUP_NAME
        print(f"Grupo destino restaurado: {original_group_name} -> {TARGET_GROUP_NAME}")
        return target_group

    available = [layer_long_name(lyr) for lyr in direct_child_groups(map_obj, parent_group)]
    raise ValueError(
        f"No se encontro el subgrupo '{TARGET_GROUP_NAME}' bajo '{PARENT_GROUP_NAME}' "
        f"ni un grupo candidato con imagenes. Subgrupos disponibles: {available}"
    )


def add_raster_to_target_group(map_obj, group_layer, image_path, target_name):
    before_layer_names = {layer_long_name(lyr) for lyr in group_direct_raster_layers(map_obj, group_layer)}
    safe_tmp_stem = re.sub(r"[^A-Za-z0-9_]+", "_", Path(image_path).stem)
    tmp_name = f"tmp_{safe_tmp_stem}"
    tmp_layer_file_path = Path.cwd() / f"{tmp_name}.lyrx"

    arcpy.management.MakeRasterLayer(image_path, tmp_name)
    arcpy.management.SaveToLayerFile(tmp_name, str(tmp_layer_file_path), "ABSOLUTE")
    tmp_layer_file = arcpy.mp.LayerFile(str(tmp_layer_file_path))

    try:
        result = map_obj.addLayerToGroup(group_layer, tmp_layer_file, "TOP")
        added_layer = None

        if isinstance(result, list) and result:
            added_layer = result[0]
        elif result is not None:
            added_layer = result

        if added_layer is None:
            after_layers = group_direct_raster_layers(map_obj, group_layer)
            new_layers = [lyr for lyr in after_layers if layer_long_name(lyr) not in before_layer_names]
            if new_layers:
                added_layer = new_layers[0]

        if added_layer is None:
            target_keys = comparable_layer_keys(target_name)
            for lyr in group_direct_raster_layers(map_obj, group_layer):
                if target_keys.intersection(comparable_layer_keys(lyr.name)):
                    added_layer = lyr
                    break

        if added_layer is not None:
            added_layer.name = target_name

        return added_layer
    finally:
        try:
            arcpy.management.Delete(tmp_name)
        except Exception:
            pass
        try:
            tmp_layer_file_path.unlink(missing_ok=True)
        except Exception:
            pass


def move_layer_to_top_of_group(map_obj, group_layer, layer_to_move):
    current_layers = group_direct_raster_layers(map_obj, group_layer)
    if not current_layers or layer_to_move == current_layers[0]:
        return
    map_obj.moveLayer(current_layers[0], layer_to_move, "BEFORE")


def order_group_layers_newest_first(map_obj, group_layer):
    ordered_layers = sorted(
        group_direct_raster_layers(map_obj, group_layer),
        key=lambda layer: sort_key_newest_first(layer.name),
        reverse=True,
    )
    for layer in reversed(ordered_layers):
        move_layer_to_top_of_group(map_obj, group_layer, layer)


image_paths_to_add = read_loaded_image_paths(LOAD_RESULTS_CSV)
print(f"Imagenes cargadas a revisar en APRX: {len(image_paths_to_add)}")

aprx = arcpy.mp.ArcGISProject(aprx_path)

try:
    maps = aprx.listMaps(map_name)

    if not maps:
        print("No se encontro el mapa")
    else:
        m = maps[0]
        parent_group = find_group(m, PARENT_GROUP_NAME)
        target_group = find_or_restore_target_group(m, parent_group)
        print(f"Grupo padre: {layer_long_name(parent_group)}")
        print(f"Grupo destino: {layer_long_name(target_group)}")

        if not is_direct_child_of_group(target_group, parent_group):
            raise ValueError(f"Grupo destino inesperado: {layer_long_name(target_group)}")

        existing_layers_before = group_direct_raster_layers(m, target_group)
        existing_keys = set()
        for lyr in existing_layers_before:
            existing_keys.update(comparable_layer_keys(lyr.name))
        print(f"Raster directos en '{TARGET_GROUP_NAME}' antes de agregar: {len(existing_layers_before)}")

        added_count = 0
        skipped_count = 0
        renamed_count = 0
        error_count = 0

        # 1. Agregar solo las imagenes nuevas como raster directos dentro de Vuelos Drone PAO > Imagenes Drone.
        for image_path in image_paths_to_add:
            target_name = layer_name_from_path(image_path)
            target_keys = comparable_layer_keys(target_name)

            if ADD_ONLY_MISSING_TO_GROUP and target_keys.intersection(existing_keys):
                print(f"Ya existe en el grupo, se conserva y se omite: {target_name}")
                skipped_count += 1
                continue

            try:
                added_layer = add_raster_to_target_group(m, target_group, image_path, target_name)
                if added_layer is None:
                    print(f"Advertencia: no se pudo confirmar el objeto retornado, se continuara con validacion posterior: {target_name}")
                existing_keys.update(target_keys)
                added_count += 1
                print(f"Agregada al grupo: {target_name}")
            except Exception as exc:
                error_count += 1
                print(f"Error agregando {image_path}: {exc}")

        # 2. Renombrar raster directos al formato corto: YY_MM_DD_Sector, sin prefijo, sin tmp_ y sin .tif.
        for lyr in group_direct_raster_layers(m, target_group):
            try:
                original_name = lyr.name
                new_name = short_layer_name(original_name)
                if new_name != original_name:
                    print(f"Renombrando: {original_name} -> {new_name}")
                    lyr.name = new_name
                    renamed_count += 1
            except Exception as exc:
                error_count += 1
                print(f"Error con capa {lyr.name}: {exc}")

        # 3. Ordenar de mas nueva a mas vieja dentro de Imagenes Drone.
        order_group_layers_newest_first(m, target_group)

        layers_after = group_direct_raster_layers(m, target_group)
        print(f"Raster directos en '{TARGET_GROUP_NAME}' despues de agregar: {len(layers_after)}")
        print(f"Resumen APRX: nuevas_agregadas={added_count}, existentes_conservadas={skipped_count}, renombradas={renamed_count}, errores={error_count}")

        if SAVE_COPY_FOR_REVIEW:
            aprx.saveACopy(APRX_COPY_PATH)
            print(f"Copia de verificacion guardada: {APRX_COPY_PATH}")
        else:
            aprx.save()
            print("Proyecto original guardado")
finally:
    del aprx


In [ ]:
from pathlib import PurePath
PurePath(image_paths_to_add[0])

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import shutil
import arcpy

LOAD_RESULTS_CSV = Path.cwd() / "outputs" / "carga_mosaico" / "20260615_162625" / "01_load_results.csv"
PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

GEOMETRY_SOURCE = "FEATURE_CLASS_ARCPY"  # FEATURE_CLASS_ARCPY o MOSAIC_DATASET
PATH_FC_FOOTPRINTS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
FOOTPRINT_NAME_FIELD = "Name"

NORMALIZED_OUTPUT_ROOT = Path.cwd() / "outputs" / "normalizacion_footprintV7" 

NORMALIZE_METHOD = "RASTERIO_MASK"
REPLACE_ORIGINALS = False
CREATE_ORIGINAL_BACKUP = True
BUILD_PYRAMIDS_AND_STATS = True

arcpy.env.overwriteOutput = True


def read_loaded_rows(csv_path):
    with csv_path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = csv.DictReader(file)
        return [row for row in rows if row.get("overall_status") == "ok" and row.get("destination_path")]


def quote_sql_text(value):
    return str(value).replace("'", "''")


def name_where_clause(dataset, field_name, name):
    field = arcpy.AddFieldDelimiters(dataset, field_name)
    return f"{field} = '{quote_sql_text(name)}'"


def resolve_geometry_source():
    source = str(GEOMETRY_SOURCE).strip()
    if source in {"FEATURE_CLASS_ARCPY", "FEATURE_CLASS_GEOPANDAS", "MOSAIC_DATASET"}:
        if source == "FEATURE_CLASS_GEOPANDAS":
            # El ambiente ArcGIS Pro puede tener pandas/numpy incompatibles. Usamos ArcPy para leer el FC.
            source = "FEATURE_CLASS_ARCPY"
        return source, PATH_FC_FOOTPRINTS

    if source and source not in {"FEATURE_CLASS_ARCPY", "FEATURE_CLASS_GEOPANDAS", "MOSAIC_DATASET"}:
        return "FEATURE_CLASS_ARCPY", source

    return source, PATH_FC_FOOTPRINTS


def raster_clip_output_path(source_path, output_root):
    source = Path(source_path)
    datastore_parent = source.parent.name
    output_dir = output_root / datastore_parent
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / source.name


def arcpy_geometry_to_geojson_geometry(geometry):
    if hasattr(geometry, "__geo_interface__"):
        return geometry.__geo_interface__
    converted = arcpy.AsShape(json.loads(geometry.JSON), True)
    if hasattr(converted, "__geo_interface__"):
        return converted.__geo_interface__
    raise RuntimeError("No se pudo convertir la geometria ArcPy a GeoJSON")


def load_footprints_index(feature_class, name_field):
    if not feature_class:
        raise ValueError("PATH_FC_FOOTPRINTS esta vacio. Define la ruta al feature class de footprints.")

    fields = [field.name for field in arcpy.ListFields(feature_class)]
    if name_field not in fields:
        raise ValueError(f"El campo '{name_field}' no existe en {feature_class}. Campos disponibles: {fields}")

    index = {}
    duplicates = set()
    with arcpy.da.SearchCursor(feature_class, [name_field, "SHAPE@"]) as cursor:
        for name, geometry in cursor:
            if not name or not geometry:
                continue
            key = str(name).lower()
            if key in index:
                duplicates.add(str(name))
            index.setdefault(key, []).append(geometry)

    if duplicates:
        print(f"Advertencia: hay nombres duplicados en footprints: {sorted(duplicates)[:10]}")
    if not index:
        raise RuntimeError(f"No se encontraron geometrías en {feature_class}")

    print(f"Footprints cargados con ArcPy: {sum(len(v) for v in index.values())} desde {feature_class}")
    return index


def footprint_geometry_from_index(footprints_index, raster_name, target_spatial_reference):
    matches = footprints_index.get(str(raster_name).lower(), [])
    if len(matches) != 1:
        raise RuntimeError(f"Footprint esperado 1 en feature class, encontrado {len(matches)}, Name={raster_name}")

    geometry = matches[0]
    if target_spatial_reference and geometry.spatialReference and geometry.spatialReference.factoryCode != target_spatial_reference.factoryCode:
        geometry = geometry.projectAs(target_spatial_reference)

    return arcpy_geometry_to_geojson_geometry(geometry)


def selected_footprint_geometry_from_mosaic(mosaic_dataset, raster_name, target_spatial_reference):
    layer_name = f"footprint_{abs(hash(raster_name))}"
    where_clause = name_where_clause(mosaic_dataset, "Name", raster_name)
    arcpy.management.MakeFeatureLayer(mosaic_dataset, layer_name, where_clause)
    try:
        count = int(arcpy.management.GetCount(layer_name).getOutput(0))
        if count != 1:
            raise RuntimeError(f"Footprint esperado 1, encontrado {count}, Name={raster_name}")

        with arcpy.da.SearchCursor(layer_name, ["SHAPE@"]) as cursor:
            geometry = next(cursor)[0]

        if target_spatial_reference and geometry.spatialReference.factoryCode != target_spatial_reference.factoryCode:
            geometry = geometry.projectAs(target_spatial_reference)

        return arcpy_geometry_to_geojson_geometry(geometry)
    finally:
        arcpy.management.Delete(layer_name)


def normalize_with_rasterio_mask(source_path, footprint_geometry, output_path):
    import rasterio
    from rasterio.mask import mask

    with rasterio.open(source_path) as src:
        masked_data, out_transform = mask(src, [footprint_geometry], crop=True, filled=False)
        profile = src.profile.copy()
        profile.update(
            driver="GTiff",
            height=masked_data.shape[1],
            width=masked_data.shape[2],
            transform=out_transform,
            compress="LZW",
            BIGTIFF="IF_SAFER",
        )

        if src.count == 3 and src.nodata is None:
            rgb = masked_data.filled(0)
            alpha = (~masked_data.mask.all(axis=0)).astype("uint8") * 255
            profile.update(count=4, dtype=rgb.dtype, nodata=None)
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(rgb, indexes=[1, 2, 3])
                dst.write(alpha, indexes=4)
                dst.colorinterp = (
                    rasterio.enums.ColorInterp.red,
                    rasterio.enums.ColorInterp.green,
                    rasterio.enums.ColorInterp.blue,
                    rasterio.enums.ColorInterp.alpha,
                )
        else:
            nodata_value = src.nodata if src.nodata is not None else 0
            data = masked_data.filled(nodata_value)
            profile.update(nodata=nodata_value)
            with rasterio.open(output_path, "w", **profile) as dst:
                dst.write(data)
                if src.colorinterp:
                    dst.colorinterp = src.colorinterp


def normalize_raster_by_footprint(source_path, raster_name, output_path, footprints_index=None):
    raster_sr = arcpy.Describe(source_path).spatialReference

    if ACTIVE_GEOMETRY_SOURCE == "FEATURE_CLASS_ARCPY":
        footprint_geometry = footprint_geometry_from_index(footprints_index, raster_name, raster_sr)
    elif ACTIVE_GEOMETRY_SOURCE == "MOSAIC_DATASET":
        footprint_geometry = selected_footprint_geometry_from_mosaic(PATH_MOSAIC_DATASET, raster_name, raster_sr)
    else:
        raise ValueError(f"GEOMETRY_SOURCE no soportado: {GEOMETRY_SOURCE} / activo: {ACTIVE_GEOMETRY_SOURCE}")

    normalize_with_rasterio_mask(source_path, footprint_geometry, output_path)

    if BUILD_PYRAMIDS_AND_STATS:
        arcpy.management.BuildPyramidsandStatistics(str(output_path))


def replace_original_with_backup(original_path, normalized_path):
    original = Path(original_path)
    normalized = Path(normalized_path)
    backup = original.with_suffix(original.suffix + ".bak_original")

    if CREATE_ORIGINAL_BACKUP and not backup.exists():
        shutil.copy2(original, backup)

    shutil.copy2(normalized, original)
    if BUILD_PYRAMIDS_AND_STATS:
        arcpy.management.BuildPyramidsandStatistics(str(original))
    return backup


rows = read_loaded_rows(LOAD_RESULTS_CSV)
NORMALIZED_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
ACTIVE_GEOMETRY_SOURCE, ACTIVE_FOOTPRINTS_PATH = resolve_geometry_source()

footprints_index = None
if ACTIVE_GEOMETRY_SOURCE == "FEATURE_CLASS_ARCPY":
    footprints_index = load_footprints_index(ACTIVE_FOOTPRINTS_PATH, FOOTPRINT_NAME_FIELD)

print(f"Imagenes a normalizar por footprint: {len(rows)}")
print(f"Metodo: {NORMALIZE_METHOD}")
print(f"Fuente geometria: {ACTIVE_GEOMETRY_SOURCE}")
print(f"Feature footprints: {ACTIVE_FOOTPRINTS_PATH}")
print(f"Salida de revision: {NORMALIZED_OUTPUT_ROOT}")
print(f"Reemplazar originales: {REPLACE_ORIGINALS}")

results = []
for row in rows:
    raster_name = row["Name"]
    source_path = row["destination_path"]
    output_path = raster_clip_output_path(source_path, NORMALIZED_OUTPUT_ROOT)

    item = {
        "Name": raster_name,
        "source_path": source_path,
        "normalized_path": str(output_path),
        "method": NORMALIZE_METHOD,
        "geometry_source": ACTIVE_GEOMETRY_SOURCE,
        "footprints_path": ACTIVE_FOOTPRINTS_PATH,
        "replace_original": REPLACE_ORIGINALS,
        "status": "pending",
        "error": "",
        "backup_path": "",
    }

    try:
        if not Path(source_path).exists():
            raise FileNotFoundError(source_path)

        normalize_raster_by_footprint(source_path, raster_name, output_path, footprints_index)

        if REPLACE_ORIGINALS:
            backup = replace_original_with_backup(source_path, output_path)
            item["backup_path"] = str(backup)
            item["status"] = "normalized_and_replaced"
        else:
            item["status"] = "normalized_for_review"

        print(f"OK {raster_name} -> {output_path}")
    except Exception as exc:
        item["status"] = "error"
        item["error"] = str(exc)
        print(f"ERROR {raster_name}: {exc}")

    results.append(item)
    

summary_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados.csv"
with summary_csv.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0].keys()) if results else ["Name", "status"])
    writer.writeheader()
    writer.writerows(results)

status_counts = {}
for item in results:
    status_counts[item["status"]] = status_counts.get(item["status"], 0) + 1

print("Resumen normalizacion:", status_counts)
print(f"CSV resultados: {summary_csv}")



In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import shutil
import arcpy

NORMALIZED_OUTPUT_ROOT = Path.cwd() / "outputs" / "normalizacion_footprintV7" 
summary_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados_V2.csv"
# Si quieres forzar una corrida especifica, pega aqui el CSV validado.
# Si queda None, toma el CSV rasterio mas reciente bajo outputs/.
NORMALIZATION_RESULTS_CSV = None
REPLACE_VALIDATED_ORIGINALS = True
CREATE_ORIGINAL_BACKUP = False
BUILD_PYRAMIDS_AFTER_REPLACE = False
REPLACE_STATUS_FILTER = {"normalized_for_review"}


def find_latest_normalization_results():
    candidates = sorted(
        (Path.cwd() / "outputs").glob("**/00_normalizacion_footprint_rasterio_resultados.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError("No se encontro ningun 00_normalizacion_footprint_rasterio_resultados.csv bajo outputs/.")
    return candidates[0]


def read_normalization_results(csv_path):
    with Path(csv_path).open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def replace_original(row):
    source_path = Path(row["source_path"])
    normalized_path = Path(row["normalized_path"])
    backup_path = source_path.with_suffix(source_path.suffix + ".bak_original")

    if not source_path.exists():
        raise FileNotFoundError(f"No existe original: {source_path}")
    if not normalized_path.exists():
        raise FileNotFoundError(f"No existe normalizado: {normalized_path}")

    if CREATE_ORIGINAL_BACKUP and not backup_path.exists():
        shutil.copy2(source_path, backup_path)

    shutil.copy2(normalized_path, source_path)

    if BUILD_PYRAMIDS_AFTER_REPLACE:
        arcpy.management.BuildPyramidsandStatistics(str(source_path))

    return backup_path

results_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados.csv"
# results_csv = Path(NORMALIZATION_RESULTS_CSV) if NORMALIZATION_RESULTS_CSV else find_latest_normalization_results()
rows = read_normalization_results(results_csv)
rows_to_replace = [row for row in rows if row.get("status") in REPLACE_STATUS_FILTER]

print(f"CSV normalizacion validado: {results_csv}")
print(f"Registros en CSV: {len(rows)}")
print(f"Registros a reemplazar: {len(rows_to_replace)}")
print(f"Reemplazo activo: {REPLACE_VALIDATED_ORIGINALS}")

replace_run_dir = results_csv.parent / f"replace_originals_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
replace_run_dir.mkdir(parents=True, exist_ok=True)
replace_log_csv = replace_run_dir / "00_replace_originals_resultados.csv"

replace_results = []
for row in rows_to_replace:
    item = {
        "Name": row.get("Name", ""),
        "source_path": row.get("source_path", ""),
        "normalized_path": row.get("normalized_path", ""),
        "backup_path": "",
        "status": "pending",
        "error": "",
    }

    try:
        if REPLACE_VALIDATED_ORIGINALS:
            backup_path = replace_original(row)
            item["backup_path"] = str(backup_path)
            item["status"] = "replaced"
            print(f"REEMPLAZADO {item['Name']}")
        else:
            item["status"] = "dry_run"
            print(f"DRY RUN {item['Name']}")
    except Exception as exc:
        item["status"] = "error"
        item["error"] = str(exc)
        print(f"ERROR {item['Name']}: {exc}")

    replace_results.append(item)

with replace_log_csv.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(replace_results[0].keys()) if replace_results else ["Name", "status"])
    writer.writeheader()
    writer.writerows(replace_results)

status_counts = {}
for item in replace_results:
    status_counts[item["status"]] = status_counts.get(item["status"], 0) + 1

print("Resumen reemplazo:", status_counts)
print(f"CSV reemplazo: {replace_log_csv}")


In [ ]:
import arcpy 
import re 
from arcgis import GIS
import pandas as pd
import numpy as np
fc_footprint = r'\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO'


In [ ]:
sdf = pd.DataFrame.spatial.from_featureclass(fc_footprint)
print(f'total de registros {sdf.shape[0]}')
sdf = sdf.sort_values(['Fecha_Adqu','Name'],ascending=[False,True]).reset_index(drop=True)
sdf = sdf[sdf['Nombre_de_Vuelo'].notnull()]

print(f'total de registros {sdf.shape[0]}')
del sdf['OBJECTID']
sdf.head()

In [ ]:
# cols_add = [c for c in sdf.columns if not re.search('shape',c,re.IGNORECASE)] + ['@SHAPE']
# with arcpy.da.InsertCursor(fc_footprint,cols_add) as cursor:
#     for i,c in df.ite

In [ ]:
PATH_MOSAIC_DATASET = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\CL MLP PAO Aereo Image Server_v2\SQLServer-amssclgis06_ArcGIS-Aereo.sde\OWD.CL_MLP_PAO_IF_Ortho_Geosupport"

names_update = [c[0] for c in arcpy.da.SearchCursor(PATH_MOSAIC_DATASET,'Name')]


In [ ]:
df_update = sdf[sdf.Name.isin(names_update)].reset_index(drop=True).copy()
df_update = df_update[['Name','Nombre_de_Vuelo']]
print(f'total de registros {df_update.shape[0]}')

In [ ]:
dictUpdate = dict(zip(df_update['Name'],df_update['Nombre_de_Vuelo']))
with arcpy.da.UpdateCursor(PATH_MOSAIC_DATASET,['NombreVuelo']) as cursor:
    for c in cursor:
        data = dictUpdate.get(c[0],None)
        if data:
            c[0] = data
            cursor.updateRow(c)
        

In [ ]:
data = dictUpdate.get(c[0],None)
data

In [ ]:
load_df.columns

In [ ]:
load_df = load_df[['path','Path_Destino']]
load_df.head()

In [ ]:
import shutil
for i,c in load_df.iterrows():
    name = Path(c['Path_Destino']).name
    # if name == 'CL_MLP_PAO_IF_Ortho_26_05_17_EV2.tif':
    #     break
    
    shutil.copy2(c['path'],c['Path_Destino'])

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import arcpy

DESTINATION_FIELD_CANDIDATES = ["Path_Destino", "destination_path"]
NAME_FIELD_CANDIDATES = ["Name", "Raster", "file_name"]

COLOR_AUDIT_OUTPUT_DIR = OUTPUT_DIR / f"audit_colormap_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
COLOR_AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COLOR_AUDIT_CSV = COLOR_AUDIT_OUTPUT_DIR / "00_audit_colormap_rasters.csv"


def first_existing_column(df, candidates):
    for column in candidates:
        if column in df.columns:
            return column
    raise ValueError(f"No se encontro ninguna columna esperada: {candidates}. Columnas disponibles: {list(df.columns)}")


def optional_column(df, candidates):
    for column in candidates:
        if column in df.columns:
            return column
    return None


def safe_get(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default


def raster_band_paths(raster_path, band_count):
    return [f"{raster_path}\\Band_{idx}" for idx in range(1, int(band_count or 0) + 1)]


def band_has_colormap(band_path):
    try:
        desc = arcpy.Describe(band_path)
        # En ArcPy, algunos drivers exponen hasColormap; otros no.
        has_colormap = safe_get(desc, "hasColormap", None)
        if has_colormap is not None:
            return bool(has_colormap), "hasColormap"

        # Respaldo: intentar leer la tabla de colormap si existe.
        colormap = safe_get(desc, "colormap", None)
        if colormap:
            return True, "colormap_property"
    except Exception as exc:
        return False, f"band_error: {exc}"

    return False, "no_colormap_property"


def inspect_raster_color(path_value):
    raster_path = str(path_value)
    item = {
        "path": raster_path,
        "exists": Path(raster_path).exists(),
        "band_count": None,
        "pixel_type": None,
        "format": None,
        "compression_type": None,
        "has_colormap": False,
        "colormap_bands": "",
        "band_notes": "",
        "is_probably_rgb": False,
        "status": "pending",
        "error": "",
    }

    try:
        if not item["exists"]:
            raise FileNotFoundError(raster_path)

        desc = arcpy.Describe(raster_path)
        item["band_count"] = safe_get(desc, "bandCount")
        item["pixel_type"] = safe_get(desc, "pixelType")
        item["format"] = safe_get(desc, "format")
        item["compression_type"] = safe_get(desc, "compressionType")

        colormap_bands = []
        notes = []
        for band_path in raster_band_paths(raster_path, item["band_count"]):
            has_colormap, note = band_has_colormap(band_path)
            notes.append(f"{Path(band_path).name}:{note}")
            if has_colormap:
                colormap_bands.append(Path(band_path).name)

        item["has_colormap"] = bool(colormap_bands)
        item["colormap_bands"] = "|".join(colormap_bands)
        item["band_notes"] = "|".join(notes)
        item["is_probably_rgb"] = int(item["band_count"] or 0) >= 3 and not item["has_colormap"]
        item["status"] = "ok"
    except Exception as exc:
        item["status"] = "error"
        item["error"] = str(exc)

    return item


destination_field = first_existing_column(load_df, DESTINATION_FIELD_CANDIDATES)
name_field = optional_column(load_df, NAME_FIELD_CANDIDATES)

audit_df = load_df[[destination_field] + ([name_field] if name_field else [])].dropna(subset=[destination_field]).copy()
print(f"Campo destino auditado: {destination_field}")
print(f"Rasters a revisar: {len(audit_df)}")

results = []
for index, row in audit_df.iterrows():
    raster_path = row[destination_field]
    item = inspect_raster_color(raster_path)
    item["Name"] = row[name_field] if name_field else Path(str(raster_path)).name
    results.append(item)

with COLOR_AUDIT_CSV.open("w", encoding="utf-8-sig", newline="") as file:
    fieldnames = ["Name", "path", "exists", "band_count", "pixel_type", "format", "compression_type", "has_colormap", "colormap_bands", "band_notes", "is_probably_rgb", "status", "error"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

colormap_rows = [row for row in results if row["has_colormap"]]
error_rows = [row for row in results if row["status"] == "error"]
not_rgb_rows = [row for row in results if row["status"] == "ok" and not row["is_probably_rgb"]]

print(f"Con colormap: {len(colormap_rows)}")
print(f"No RGB probable: {len(not_rgb_rows)}")
print(f"Errores: {len(error_rows)}")
print(f"CSV auditoria: {COLOR_AUDIT_CSV}")

if colormap_rows:
    print("Rasters con colormap:")
    for row in colormap_rows[:20]:
        print(f"- {row['Name']} | bandas: {row['colormap_bands']} | {row['path']}")


In [ ]:
dfcsv = pd.read_csv(COLOR_AUDIT_CSV)
dfcsv.head()

In [ ]:
str(COLOR_AUDIT_CSV)

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import json
import shutil

import arcpy

# Entrada: CSV de auditoria ya generado o DataFrame dfcsv ya cargado.
AUDIT_CSV_FOR_RGB_CLEAN = 'c:\\Users\\esrlrivero_adm\\Documents\\Geosupport\\amsa-pao-geosupport\\outputs\\carga_mosaico\\20260617_113254\\audit_colormap_20260617_115919\\00_audit_colormap_rasters.csv'
PATH_FC_FOOTPRINTS = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\02_FGDB\CL_MLP_PAO_v1.gdb\CL_MLP_PAO_06_COMPLEMENTOS\CL_MLP_PAO_Indice_Vuelos_PAO_IMGS_PO"
FOOTPRINT_NAME_FIELD = "Name"

RGB_CLEAN_OUTPUT_DIR = Path.cwd() / "outputs" / "rgb_clean_from_audit" / datetime.now().strftime("%Y%m%d_%H%M%S")
REPLACE_RASTER_WITH_RGB_CLEAN = True
CREATE_BACKUP_BEFORE_RGB_REPLACE = False
PROCESS_ONLY_COLORMAP_OR_NOT_RGB = True
TEST_ONLY_FIRST_IMAGE = False


def read_audit_rows():
    if "dfcsv" in globals():
        return dfcsv.to_dict("records")

    with Path(AUDIT_CSV_FOR_RGB_CLEAN).open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def truthy(value):
    return str(value).strip().lower() in {"true", "1", "yes", "si", "sí"}


def arcpy_geometry_to_geojson_geometry(geometry):
    if hasattr(geometry, "__geo_interface__"):
        return geometry.__geo_interface__
    converted = arcpy.AsShape(json.loads(geometry.JSON), True)
    if hasattr(converted, "__geo_interface__"):
        return converted.__geo_interface__
    raise RuntimeError("No se pudo convertir geometria ArcPy a GeoJSON")


def footprint_name_variants(name):
    text = str(name).strip()
    variants = [text]
    if text.lower().endswith(".tif"):
        variants.append(text[:-4])
    if text.lower().startswith("tmp_"):
        variants.append(text[4:])
    for value in list(variants):
        if value.lower().endswith(".tif"):
            variants.append(value[:-4])
    return list(dict.fromkeys([value for value in variants if value]))


def load_footprints_index(feature_class, name_field):
    fields = [field.name for field in arcpy.ListFields(feature_class)]
    if name_field not in fields:
        raise ValueError(f"El campo '{name_field}' no existe en {feature_class}. Campos disponibles: {fields}")

    index = {}
    with arcpy.da.SearchCursor(feature_class, [name_field, "SHAPE@"]) as cursor:
        for name, geometry in cursor:
            if not name or not geometry:
                continue
            for variant in footprint_name_variants(name):
                index.setdefault(variant.lower(), []).append(geometry)
    return index


def footprint_geometry_for_name(footprints_index, raster_name, target_spatial_reference):
    tried = footprint_name_variants(raster_name)
    matches = []
    matched_key = None
    for candidate in tried:
        matches = footprints_index.get(candidate.lower(), [])
        if matches:
            matched_key = candidate
            break

    if len(matches) != 1:
        raise RuntimeError(
            f"Footprint esperado 1, encontrado {len(matches)}, Name={raster_name}, "
            f"variantes_probadas={tried}"
        )

    geometry = matches[0]
    if target_spatial_reference and geometry.spatialReference and geometry.spatialReference.factoryCode != target_spatial_reference.factoryCode:
        geometry = geometry.projectAs(target_spatial_reference)
    return arcpy_geometry_to_geojson_geometry(geometry)


def output_path_for_raster(raster_path):
    raster_path = Path(raster_path)
    out_dir = RGB_CLEAN_OUTPUT_DIR / raster_path.parent.name
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / raster_path.name


def write_rgb_clean_with_rasterio(source_path, footprint_geometry, output_path):
    import numpy as np
    import rasterio
    from rasterio.mask import mask

    with rasterio.open(source_path) as src:
        if src.count >= 3:
            masked_data, out_transform = mask(
                src,
                [footprint_geometry],
                crop=True,
                filled=False,
                indexes=[1, 2, 3],
            )
            rgb = masked_data.filled(0)
        elif src.count == 1:
            try:
                colormap = src.colormap(1)
            except Exception as exc:
                raise RuntimeError(f"Raster de 1 banda sin colormap legible: {source_path}") from exc

            masked_data, out_transform = mask(
                src,
                [footprint_geometry],
                crop=True,
                filled=False,
                indexes=1,
            )
            values = masked_data.filled(0)
            rgb = np.zeros((3, values.shape[0], values.shape[1]), dtype="uint8")
            for pixel_value, color in colormap.items():
                color_mask = values == pixel_value
                if not color_mask.any():
                    continue
                rgb[0][color_mask] = color[0]
                rgb[1][color_mask] = color[1]
                rgb[2][color_mask] = color[2]
        else:
            raise RuntimeError(f"Raster sin bandas validas: {source_path}, bands={src.count}")

        profile = src.profile.copy()
        for key in [
            "alpha", "photometric", "nodata", "interleave", "compress",
            "jpeg_quality", "predictor", "blockxsize", "blockysize", "tiled",
            "count", "dtype",
        ]:
            profile.pop(key, None)

        profile.update(
            driver="GTiff",
            count=3,
            height=rgb.shape[1],
            width=rgb.shape[2],
            transform=out_transform,
            dtype="uint8" if src.count == 1 else rgb.dtype,
            nodata=0,
            compress="LZW",
            photometric="RGB",
            interleave="pixel",
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(output_path, "w", **profile) as dst:
            dst.write(rgb.astype(profile["dtype"], copy=False))
            dst.colorinterp = (
                rasterio.enums.ColorInterp.red,
                rasterio.enums.ColorInterp.green,
                rasterio.enums.ColorInterp.blue,
            )


def replace_with_backup(source_path, clean_path):
    source_path = Path(source_path)
    clean_path = Path(clean_path)
    backup_path = source_path.with_suffix(source_path.suffix + ".backup_before_rgb_clean")
    if CREATE_BACKUP_BEFORE_RGB_REPLACE and not backup_path.exists():
        shutil.copy2(source_path, backup_path)
    shutil.copy2(clean_path, source_path)
    return backup_path


audit_rows = read_audit_rows()
required = {"Name", "path"}
available_columns = set(audit_rows[0].keys()) if audit_rows else set()
missing = required.difference(available_columns)
if missing:
    raise ValueError(f"Faltan columnas en auditoria: {missing}. Columnas: {sorted(available_columns)}")

work_rows = list(audit_rows)
if PROCESS_ONLY_COLORMAP_OR_NOT_RGB:
    work_rows = [
        row for row in work_rows
        if truthy(row.get("has_colormap", False)) or not truthy(row.get("is_probably_rgb", True))
    ]

if TEST_ONLY_FIRST_IMAGE:
    work_rows = work_rows[:1]

RGB_CLEAN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
footprints_index = load_footprints_index(PATH_FC_FOOTPRINTS, FOOTPRINT_NAME_FIELD)
print(f"Rasters a normalizar RGB: {len(work_rows)}")
print(f"Salida RGB limpia: {RGB_CLEAN_OUTPUT_DIR}")
print(f"Reemplazar original: {REPLACE_RASTER_WITH_RGB_CLEAN}")
print(f"Solo primera prueba: {TEST_ONLY_FIRST_IMAGE}")

results = []
for row in work_rows:
    raster_name = row["Name"]
    raster_path = row["path"]
    clean_path = output_path_for_raster(raster_path)
    item = {
        "Name": raster_name,
        "source_path": raster_path,
        "rgb_clean_path": str(clean_path),
        "replace_original": REPLACE_RASTER_WITH_RGB_CLEAN,
        "backup_path": "",
        "status": "pending",
        "error": "",
    }

    try:
        raster_sr = arcpy.Describe(raster_path).spatialReference
        footprint_geometry = footprint_geometry_for_name(footprints_index, raster_name, raster_sr)
        write_rgb_clean_with_rasterio(raster_path, footprint_geometry, clean_path)

        if REPLACE_RASTER_WITH_RGB_CLEAN:
            item["backup_path"] = str(replace_with_backup(raster_path, clean_path))
            item["status"] = "rgb_clean_replaced"
        else:
            item["status"] = "rgb_clean_for_review"
        print(f"OK {raster_name} -> {clean_path}")
    except Exception as exc:
        item["status"] = "error"
        item["error"] = str(exc)
        print(f"ERROR {raster_name}: {exc}")

    results.append(item)

result_csv = RGB_CLEAN_OUTPUT_DIR / "00_rgb_clean_from_audit_resultados.csv"
with result_csv.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0].keys()) if results else ["Name", "status"])
    writer.writeheader()
    writer.writerows(results)

status_counts = {}
for item in results:
    status_counts[item["status"]] = status_counts.get(item["status"], 0) + 1
print("Resumen RGB clean:", status_counts)
print(f"CSV resultado: {result_csv}")





In [ ]:
from pathlib import Path
import pandas as pd
NORMALIZED_OUTPUT_ROOT = Path.cwd() / "outputs" / "normalizacion_footprintV7" 
summary_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados.csv"
csv_result = r'c:\Users\esrlrivero_adm\Documents\Geosupport\amsa-pao-geosupport\outputs\rgb_clean_from_audit\20260617_123824\00_rgb_clean_from_audit_resultados.csv'

In [ ]:
df = pd.read_csv(csv_result)
df['Name'] = df['Name'].str.replace(".tif", "", regex=False)
df.head()

In [ ]:
df2 = pd.read_csv(summary_csv)
print(f'Total de registros {df2.shape[0]}')
df2 = df2[~df2.Name.isin(df.Name)]
print(f'Total de registros {df2.shape[0]}')
df2.head()

In [ ]:
csv = f'{Path(summary_csv).parent}\{Path(summary_csv).stem}_V2.csv'
df2.to_csv(csv,index=False)

In [ ]:
df2 = pd.read_csv(csv)
df2.shape[0]

In [ ]:
df2 = df2[['footprints_path','source_path']]

In [ ]:
df2.head()

In [ ]:
from pathlib import Path
from datetime import datetime
import csv
import shutil
import arcpy

NORMALIZED_OUTPUT_ROOT = Path.cwd() / "outputs" / "normalizacion_footprintV7" 
summary_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados_V2.csv"
# Si quieres forzar una corrida especifica, pega aqui el CSV validado.
# Si queda None, toma el CSV rasterio mas reciente bajo outputs/.
NORMALIZATION_RESULTS_CSV = None
REPLACE_VALIDATED_ORIGINALS = True
CREATE_ORIGINAL_BACKUP = False
BUILD_PYRAMIDS_AFTER_REPLACE = False
REPLACE_STATUS_FILTER = {"normalized_for_review"}


def find_latest_normalization_results():
    candidates = sorted(
        (Path.cwd() / "outputs").glob("**/00_normalizacion_footprint_rasterio_resultados.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        raise FileNotFoundError("No se encontro ningun 00_normalizacion_footprint_rasterio_resultados.csv bajo outputs/.")
    return candidates[0]


def read_normalization_results(csv_path):
    with Path(csv_path).open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def replace_original(row):
    source_path = Path(row["source_path"])
    normalized_path = Path(row["normalized_path"])
    backup_path = source_path.with_suffix(source_path.suffix + ".bak_original")

    if not source_path.exists():
        raise FileNotFoundError(f"No existe original: {source_path}")
    if not normalized_path.exists():
        raise FileNotFoundError(f"No existe normalizado: {normalized_path}")

    if CREATE_ORIGINAL_BACKUP and not backup_path.exists():
        shutil.copy2(source_path, backup_path)

    shutil.copy2(normalized_path, source_path)

    if BUILD_PYRAMIDS_AFTER_REPLACE:
        arcpy.management.BuildPyramidsandStatistics(str(source_path))

    return backup_path

results_csv = NORMALIZED_OUTPUT_ROOT / "00_normalizacion_footprint_resultados_V2.csv"
# results_csv = Path(NORMALIZATION_RESULTS_CSV) if NORMALIZATION_RESULTS_CSV else find_latest_normalization_results()
rows = read_normalization_results(results_csv)
rows_to_replace = [row for row in rows if row.get("status") in REPLACE_STATUS_FILTER]

print(f"CSV normalizacion validado: {results_csv}")
print(f"Registros en CSV: {len(rows)}")
print(f"Registros a reemplazar: {len(rows_to_replace)}")
print(f"Reemplazo activo: {REPLACE_VALIDATED_ORIGINALS}")

replace_run_dir = results_csv.parent / f"replace_originals_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
replace_run_dir.mkdir(parents=True, exist_ok=True)
replace_log_csv = replace_run_dir / "00_replace_originals_resultados.csv"

replace_results = []
for row in rows_to_replace:
    item = {
        "Name": row.get("Name", ""),
        "source_path": row.get("source_path", ""),
        "normalized_path": row.get("normalized_path", ""),
        "backup_path": "",
        "status": "pending",
        "error": "",
    }

    try:
        if REPLACE_VALIDATED_ORIGINALS:
            backup_path = replace_original(row)
            item["backup_path"] = str(backup_path)
            item["status"] = "replaced"
            print(f"REEMPLAZADO {item['Name']}")
        else:
            item["status"] = "dry_run"
            print(f"DRY RUN {item['Name']}")
    except Exception as exc:
        item["status"] = "error"
        item["error"] = str(exc)
        print(f"ERROR {item['Name']}: {exc}")

    replace_results.append(item)

with replace_log_csv.open("w", encoding="utf-8-sig", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(replace_results[0].keys()) if replace_results else ["Name", "status"])
    writer.writeheader()
    writer.writerows(replace_results)

status_counts = {}
for item in replace_results:
    status_counts[item["status"]] = status_counts.get(item["status"], 0) + 1

print("Resumen reemplazo:", status_counts)
print(f"CSV reemplazo: {replace_log_csv}")


In [ ]:
CL_MLP_PAO_IF_Ortho_26_05_17_EV2.tif